# BrainDecode: Analysis Dependencies Generation Pipeline (FragPipe)

Generates all required dependency files per experiment from **FragPipe** output and
pipeline-derived dictionaries.

Outputs (saved to `DEPDIR/{experiment}/`):

1. `dataset_metrics.xlsx`
2. `Position_probability_fragment_ion_data.csv`
3. `Fragment_ion_dict.p`
4. `SAAP_precursor_reporter_quant.xlsx`
5. `MTP_sequences.fasta`
6. `Modified_peptide_filter_dict_DP2valSAAP.p`
7. `PTM_heatmap_dict.p`
8. `PTM_heatmap_data.xlsx`
9. `fragments_per_saap_4barplot_allDS.xlsx`

FragPipe notes:
- Reporter/fragment data come from `psm.tsv` and the per-fraction MSFragger `.tsv` files
  (fragments encoded `nterm{n}`=b, `cterm{n}`=y), not from `msms.txt`/`evidence.txt`.
- Fragment masses are recomputed from sequence + `modification_info` (psm.tsv has no Masses col).
- FragPipe `Probability` (higher=better) is mapped to a PEP-like score as `1 - Probability`.

Last Updated 06-24-2026 by Alex Maropakis


## Imports

In [40]:
print("Reading in packages...")

import os
import pickle
from collections import Counter
from typing import Any
import numpy as np
import pandas as pd
import re as _re
from glob import glob 

print("Packages loaded successfully!")


Reading in packages...
Packages loaded successfully!


## Directories

In [41]:
print("Loading directories...")

## Set base directories 
CODE_DIR        = '/Users/alexmaropakis/Projects/BrainDecode/'
PROJECT_DIR     = '/Users/alexmaropakis/Projects/Project_BrainDecode/'
INDIR           = PROJECT_DIR + 'Analysis_Inputs/'
OUTDIR          = CODE_DIR + 'Dependencies/Analysis_Outputs/'
DEPDIR          = CODE_DIR + 'Dependencies/'
FRAG_DIR        = PROJECT_DIR + 'frag_output/'
SAMPLE_MAPS_DIR = DEPDIR + 'Sample_maps/'

os.makedirs(DEPDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

## Ping 2018 
# Anterior cingulate gyrus 
ping2018        = INDIR + 'Ping_2018/'
acg             = ping2018 + 'acg/'
acg_frag        = FRAG_DIR + 'Ping_2018/acg/'
acg_batches     = ['b1','b2','b3','b4','b5']
acg_dirs, acg_samples, acg_maps, acg_frag_dirs = [], [], [], []
for b in acg_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_acg{b}.xlsx')
    acg_dirs.append(acg + f'{b}/')
    acg_maps.append(_map)
    acg_samples.append(['S' + str(i) for i in sorted(set(_map['TMT plex']))])
    acg_frag_dirs.append(acg_frag + f'{b}/acg{b}_1/')

# frontal cortex 
fc             = ping2018 + 'fc/'
fc_frag        = FRAG_DIR + 'Ping_2018/fc/'
fc_batches     = ['b1','b2','b3','b4','b5']
fc_dirs, fc_samples, fc_maps, fc_frag_dirs = [], [], [], []
for b in fc_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_fc{b}.xlsx')
    fc_dirs.append(fc + f'{b}/')
    fc_maps.append(_map)
    fc_samples.append(['S' + str(i) for i in sorted(set(_map['TMT plex']))])
    fc_frag_dirs.append(fc_frag + f'{b}/fc{b}_1/')

## Takasugi 2024 
ms              = INDIR + 'Takasugi_2024/'
ms_frag         = FRAG_DIR + 'Takasugi_2024/'
ms_tissues      = ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Skin']
ms_dirs, ms_samples, ms_maps, ms_frag_dirs = [], [], [], []
for tissue in ms_tissues:
    _aas_dir = ms + f'{tissue}/'
    _map     = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_{tissue.lower()}.xlsx')
    ms_dirs.append(_aas_dir)
    ms_maps.append(_map)
    ms_samples.append(['S' + str(i) for i in sorted(set(_map['TMT plex']))])
    ms_frag_dirs.append(ms_frag + f'{tissue}/{tissue.lower()}_1/')

## Bai 2020 
bai2020             = INDIR + 'Bai_2020/'
bai_frag            = FRAG_DIR + 'Bai_2020/pooled_1/'
bai_sample_map      = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_pooled.xlsx')
bai_samples         = ['S'+str(i) for i in list(set(bai_sample_map['TMT plex']))]
bai_tissue          = ['pooled']


## Tsumagari 2023 
tsumagari2023       = INDIR + 'Tsumagari_2023/'
tsumagari_frag      = FRAG_DIR + 'Tsumagari_2023/'
tsumagari_tissues    = ['cortex_1', 'cortex_2', 'hippocampus_1', 'hippocampus_2']
tsumagari_dirs, tsumagari_samples, tsumagari_maps, tsumagari_frag_dirs = [], [], [], []
for tissue in tsumagari_tissues:
    _aas_dir = tsumagari2023 + f'{tissue}/'
    _map     = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_{tissue}_tsumagari.xlsx')
    tsumagari_dirs.append(_aas_dir)
    tsumagari_maps.append(_map)
    tsumagari_samples.append(['S' + str(i) for i in sorted(set(_map['TMT plex']))])
    tsumagari_frag_dirs.append(tsumagari_frag + f'{tissue}/{tissue}_tsumagari_1/')

## Lists to easily call data
datasets = (
    [f'acg{b}' for b in acg_batches],
    [f'fc{b}'  for b in fc_batches],
    ms_tissues,
    bai_tissue,
    tsumagari_tissues,
)
all_datasets    = [d for grp in datasets for d in grp]

proj_dir_list   = acg_dirs       + fc_dirs       + ms_dirs      + [bai2020] + tsumagari_dirs
frag_dir_list   = acg_frag_dirs  + fc_frag_dirs  + ms_frag_dirs + [bai_frag] + tsumagari_frag_dirs
samples_list    = acg_samples    + fc_samples    + ms_samples   + [bai_samples] + tsumagari_samples
sample_map_list = acg_maps       + fc_maps       + ms_maps      + [bai_sample_map] + tsumagari_maps

## TMT-set to dataset name mapping 
tmt_to_dataset = {
    # Ping et al., 2018 
    '/acg/b1/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b2/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b3/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b4/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b5/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/fc/b1/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b2/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b3/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b4/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b5/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},

    # Bai et al., 2020 
    '/Bai_2020/': {'S1': 'pooled'},

    # Takasugi et al., 2024
    '/takasugi_2024/aorta/':  {'S1': 'Aorta'},
    '/takasugi_2024/brain/':  {'S2': 'Brain'},
    '/takasugi_2024/heart/':  {'S3': 'Heart'},
    '/takasugi_2024/kidney/': {'S4': 'Kidney'},
    '/takasugi_2024/liver/':  {'S5': 'Liver'},
    '/takasugi_2024/lung/':   {'S6': 'Lung'},
    '/takasugi_2024/muscle/': {'S7': 'Muscle'},
    '/takasugi_2024/skin/':   {'S8': 'Skin'},

    # Tsumagari et al., 2023
    '/tsumagari_2023/hippocampus_1/':    {'S1': 'hippocampus_1'},
    '/tsumagari_2023/hippocampus_2/':    {'S2': 'hippocampus_2'},
    '/tsumagari_2023/cortex_1/':        {'S3': 'cortex_1'},
    '/tsumagari_2023/cortex_2/':        {'S4': 'cortex_2'},
}

## Sample-type classifiers
SAMPLE_TYPES = [

    # Takasugi et al., 2024 & Tsumagari et al., 2023 
    ('t03mo', 't03mo'),
    ('t06mo', 't06mo'),
    ('t15mo', 't15mo'),
    ('t24mo', 't24mo'),
    ('t30mo', 't30mo'),

    # Bai et al., 2020 
    ('LPC_Pool1', 'LPC_Pool1'),
    ('LPC_Pool2', 'LPC_Pool2'),
    ('HPC_Pool1', 'HPC_Pool1'),
    ('HPC_Pool2', 'HPC_Pool2'),
    ('MCI_Pool1', 'MCI_Pool1'),
    ('MCI_Pool2', 'MCI_Pool2'),
    ('AD_Pool1', 'AD_Pool1'),
    ('AD_Pool2', 'AD_Pool2'),
    ('PSP_Pool1', 'PSP_Pool1'),
    ('PSP_Pool2', 'PSP_Pool2'),

    # Ping et al., 2018
    ('ADPD', 'ADPD'),
    ('AD',   'AD'),
    ('PD',   'PD'),
    ('CTRL', 'CTRL'),
]

print("Directories loaded successfully!")
print(f"  Datasets registered: {datasets}")

Loading directories...
Directories loaded successfully!
  Datasets registered: (['acgb1', 'acgb2', 'acgb3', 'acgb4', 'acgb5'], ['fcb1', 'fcb2', 'fcb3', 'fcb4', 'fcb5'], ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Skin'], ['pooled'], ['cortex_1', 'cortex_2', 'hippocampus_1', 'hippocampus_2'])


## Helper Functions

In [42]:
class CompatUnpickler(pickle.Unpickler):
    # Pickle compatibility shim for legacy numpy/pandas pickles 
    # This makes it so the Python package versions on HPC are irrelevant
    REMAP = {
        'numpy.core._multiarray_umath': 'numpy._core._multiarray_umath',
        'numpy.core.multiarray': 'numpy._core.multiarray',
        'numpy.core.numeric': 'numpy._core.numeric',
        'numpy.core': 'numpy._core',
        'pandas.core.indexes.numeric': 'pandas.core.indexes.base',
        'pandas.core.indexes.int64': 'pandas.core.indexes.base',
        'pandas.core.indexes.float64': 'pandas.core.indexes.base',
    }
    def find_class(self, module, name):
        # Intercept every class lookup during unpickling to rewrite 
        # the module path if in REMAP
        return super().find_class(self.REMAP.get(module, module), name)

def compat_load(path):
    # Function to load pickle through compatibility shim
    # Use this if reading legacy .p files
    with open(path, 'rb') as f:
        return CompatUnpickler(f).load()

## Functions to parse experiment and sample type labels 
EXPERIMENT_PREFIXES   = {'acg': 'Ping_2018/acg', 'fc': 'Ping_2018/fc', 'pooled': 'Bai_2020'}
DATASET_EXP_OVERRIDES = {**{t: 'Takasugi_2024'  for t in ms_tissues},
                         **{t: 'Tsumagari_2023' for t in tsumagari_tissues}}

def get_experiment_label(dataset_name):
    # Function to resolve dataset name to experiment group label 
    if dataset_name in DATASET_EXP_OVERRIDES:
        return DATASET_EXP_OVERRIDES[dataset_name]
    for prefix, experiment in EXPERIMENT_PREFIXES.items():
        if dataset_name.lower().startswith(prefix):
            return experiment
    return 'other'

def classify_sample(sample_name):
    # Function to sort sample into known sample types by substrip match
    for pattern, label in SAMPLE_TYPES:
        if pattern in sample_name:
            return label
    return 'Unknown'

def exp_dir(experiment):
    # Function to return/create on-disk output directory for experiment 
    path = os.path.join(DEPDIR, experiment)
    os.makedirs(path, exist_ok=True)
    return path

def resolve_dataset_key(proj):
    # Function to match dataset to TMT set (from sample map)
    d = proj.lower()
    for key in tmt_to_dataset:
        if key.lower() in d:
            return tmt_to_dataset[key]
    return {}


## FragPipe Helpers
_AA_MONO = { # Monoisotopic residue masses 
 'G':57.02146,'A':71.03711,'S':87.03203,'P':97.05276,'V':99.06841,'T':101.04768,
 'C':103.00919,'L':113.08406,'I':113.08406,'N':114.04293,'D':115.02694,'Q':128.05858,
 'K':128.09496,'E':129.04259,'M':131.04049,'H':137.05891,'F':147.06841,'R':156.10111,
 'Y':163.06333,'W':186.07931,
}
_PROTON = 1.007276 # charge-carrier mass added per ion 
_H20    = 18.010565 # mass of water (y-ion retains C-terminal OH + H)

def parse_mods(mod_str):
    # Function to parse modification_info strings into:
        # nterm = total mass shift on peptide N-terminus
        # site = dict mapping 1-based peptide residue positions to total mass shift at that residue
    nterm = 0.0
    site  = {}
    # Treat empty mod fields as no mods 
    if not isinstance(mod_str, str) or not mod_str.strip():
        return nterm, site
    for tok in mod_str.split(','):
        tok = tok.strip()
        # N-terminal mod, e.g. 'N-term(229.1629)' (TMT label on N-term)
        m = _re.match(r'N-term\(([\d.]+)\)', tok)
        if m:
            nterm += float(m.group(1)) # accumulate if multiple N-term mods
            continue
        # Residue mod, e.g. '5K(229.1629)' (position 5 +229.1629)
        m = _re.match(r'(\d+)[A-Z]\(([\d.]+)\)', tok)
        if m:
            p = int(m.group(1))
            site[p] = site.get(p, 0.0) + float(m.group(2))
    return nterm, site

def frag_ion_mass(pep, mod_str, ion):
    # Function to compute the theoretical singly charged fragment ion m/z
    # for a b or y ion, taking modifications into account
    # ion = charge-stripped token (e.g. nterm3 = b3, cterm5=y5)
    nterm, site = parse_mods(mod_str)
    if ion.startswith('nterm'):
        # b ion = first n residues from N-terminus
        n = int(ion[5:])
        base = sum(_AA_MONO[a] for a in pep[:n])
        # b-ion carries N-term mod + site mods at positions <= n
        mods = nterm + sum(m for p, m in site.items() if p <= n)
        return base + mods + _PROTON
    else:
        # y-ion = last n residues from C-terminus
        n = int(ion[5:])
        lo = len(pep) - n + 1
        base = sum(_AA_MONO[a] for a in pep[len(pep)-n:])
        # only site mods within C-terminal sppan [lo, len] belong to this y-ion
        mods = sum(m for p, m in site.items() if p >= lo)
        if lo == 1:
            mods += nterm # y only includes nterm when it spans residue 1
        return base + mods + _H20 + _PROTON 
        
def resolve_ion(frag_token):
    # Function to drop charge and resolve nterm = b and cterm = y
    # e.g. nterm3^1 = b3, cterm5^2 = y5
    ion = frag_token.split('^')[0]
    if ion.startswith('nterm'):
        return 'b' + ion[5:]
    if ion.startswith('cterm'):
        return 'y' + ion[5:]
    return ion

def load_frag_index(frag_dir):
    # Build {peptide: best-hit dict} across all per-fraction *.tsv in frag_dir.
    # This is the fast path: it precomputes the best hit per peptide ONCE so the
    # generation cell does O(1) lookups instead of scanning every row per SAAP.
    # Returns plain dicts (not pandas Series) and reads only the needed columns.
    report = {'psm', 'ion', 'peptide', 'protein'}  # FragPipe report TSVs -> skip, not per-fraction
    keep   = ['peptide', 'modification_info', 'fragments', 'num_matched_ions', 'hit_rank']
    best = {}
    for f in glob(frag_dir + '*.tsv'):
        stem = os.path.splitext(os.path.basename(f))[0]
        if stem in report:
            continue  # skip the aggregate report files; we only want per-fraction PSM TSVs
        # usecols lambda -> read only the 5 needed columns, keeps memory/time down
        d = pd.read_csv(f, sep='\t', low_memory=False, usecols=lambda c: c in keep)
        if 'fragments' not in d.columns:
            continue  # file lacks fragment data -> not a usable per-fraction TSV
        d = d[d['hit_rank'] == 1]  # rank-1 PSM only -- the engine's best ID for that scan
        # pull to numpy arrays once -> iterating these is much faster than iterrows
        peps = d['peptide'].to_numpy()
        mods = d['modification_info'].to_numpy()
        frgs = d['fragments'].to_numpy()
        nmi  = d['num_matched_ions'].to_numpy()
        for p, m, fr, n in zip(peps, mods, frgs, nmi):
            # keep, per peptide, the PSM with the most matched ions (best evidence)
            if p not in best or n > best[p]['num_matched_ions']:
                best[p] = {'modification_info': m, 'fragments': fr, 'num_matched_ions': n}
    return best

# Generate Analysis Dependencies

## 1. `dataset_metrics.xlsx`

In [43]:
print("=== Generating: dataset_metrics.xlsx ===")

rows = []
for ds, frag_dir in zip(all_datasets, frag_dir_list):
    # FragPipe: psm.tsv (PSM-level, Philosopher-FDR-filtered) + peptide.tsv
    psm     = pd.read_csv(frag_dir + 'psm.tsv',     sep='\t', low_memory=False)
    peptide = pd.read_csv(frag_dir + 'peptide.tsv', sep='\t', low_memory=False)

    n_psm        = len(psm)                              # already 1% FDR by Philosopher
    n_peptides   = peptide['Peptide'].nunique()
    n_pep_psm    = psm['Peptide'].nunique()              # peptides seen at PSM level
    n_unique     = psm.loc[psm['Is Unique'] == True, 'Peptide'].nunique()

    rows.append([ds, get_experiment_label(ds), n_psm, n_peptides, n_pep_psm, n_unique])
    print(f"  {ds}: {n_psm:,} PSMs | {n_peptides:,} peptides")

metrics_all = pd.DataFrame(rows, columns=[
    'Dataset', 'Experiment', 'Identified PSMs (1% FDR)',
    'Peptides', 'Peptides (evidence)', 'Unique peptides',
])

for exp, grp in metrics_all.groupby('Experiment'):
    out = os.path.join(exp_dir(exp), 'dataset_metrics.xlsx')
    grp.drop(columns='Experiment').reset_index(drop=True).to_excel(
        out, index=False, engine='openpyxl')
    print(f"  Saved \u2192 {out}")


=== Generating: dataset_metrics.xlsx ===
  acgb1: 209,933 PSMs | 90,703 peptides
  acgb2: 195,200 PSMs | 91,495 peptides
  acgb3: 200,265 PSMs | 94,334 peptides
  acgb4: 202,734 PSMs | 88,983 peptides
  acgb5: 194,238 PSMs | 89,149 peptides
  fcb1: 131,555 PSMs | 73,129 peptides
  fcb2: 127,955 PSMs | 68,635 peptides
  fcb3: 121,410 PSMs | 65,132 peptides
  fcb4: 147,136 PSMs | 81,523 peptides
  fcb5: 114,870 PSMs | 59,681 peptides
  Aorta: 119,924 PSMs | 58,862 peptides
  Brain: 149,476 PSMs | 67,819 peptides
  Heart: 82,879 PSMs | 45,721 peptides
  Kidney: 107,513 PSMs | 59,778 peptides
  Liver: 135,546 PSMs | 57,940 peptides
  Lung: 102,188 PSMs | 60,038 peptides
  Muscle: 63,183 PSMs | 27,272 peptides
  Skin: 141,222 PSMs | 72,645 peptides
  pooled: 2,351,055 PSMs | 284,164 peptides
  cortex_1: 143,965 PSMs | 75,853 peptides
  cortex_2: 105,869 PSMs | 61,888 peptides
  hippocampus_1: 151,204 PSMs | 83,368 peptides
  hippocampus_2: 147,110 PSMs | 81,390 peptides
  Saved → /Users/ale

## 2 + 3. `Position_probability_fragment_ion_data.csv` and `Fragment_ion_dict.p`

For each SAAP/BP pair, looks up the MS/MS spectrum with the most fragment matches
and records the four boundary b/y ions (and their masses) flanking the
substituted residue. Per-experiment positional-probability CSV and a
fragment-mass dictionary are saved together since they are built in the same
pass.


In [44]:
print("=== Generating: Position_probability_fragment_ion_data.csv + Fragment_ion_dict.p ===")

POS_COLS = [
    'SAAP', 'BP', 'AAS', 'AAS index', 'Positional probability', 'charge',
    'TMT set', 'Dataset',
    'saap_b_left_frag',  'saap_b_left_frag_mass',
    'saap_b_right_frag', 'saap_b_right_frag_mass',
    'saap_y_left_frag',  'saap_y_left_frag_mass',
    'saap_y_right_frag', 'saap_y_right_frag_mass',
    'bp_b_left_frag',    'bp_b_left_frag_mass',
    'bp_b_right_frag',   'bp_b_right_frag_mass',
    'bp_y_left_frag',    'bp_y_left_frag_mass',
    'bp_y_right_frag',   'bp_y_right_frag_mass',
]
FRAG_NAME_COLS = [c for c in POS_COLS if c.endswith('_frag')]
FRAG_MASS_COLS = [c for c in POS_COLS if c.endswith('_frag_mass')]

exp_rows, exp_frag, exp_count = {}, {}, {}

for ds, proj, samples, frag_dir in zip(all_datasets, proj_dir_list, samples_list, frag_dir_list):
    exp = get_experiment_label(ds)
    exp_rows.setdefault(exp, [])
    exp_frag.setdefault(exp, {})
    exp_count.setdefault(exp, 0)

    mtp_dict = compat_load(proj + 'Ion_validated_MTP_dict.p')
    # FragPipe: index per-fraction fragment TSVs by peptide once per plex (O(1) lookups)
    frag_by_pep = load_frag_index(frag_dir)

    for s in samples:
        if s not in mtp_dict:
            continue
        s_df = pd.DataFrame.from_dict(mtp_dict[s])[
            ['mistranslated sequence', 'DP Base Sequence', 'aa subs',
             'origin aa index', 'aa subs positional probability', 'Charge']
        ].copy()
        s_df['TMT set'] = s
        s_df['Dataset'] = ds
        s_df.reset_index(drop=True, inplace=True)
        for c in FRAG_NAME_COLS: s_df[c] = ''
        for c in FRAG_MASS_COLS: s_df[c] = np.nan

        for i, row in s_df.iterrows():
            saap, bp, idx = row['mistranslated sequence'], row['DP Base Sequence'], row['origin aa index']
            global_i = i + exp_count[exp]
            exp_frag[exp][global_i] = {}

            for peptide, mapping, frag_key in [
                (saap, [
                    ('b' + str(idx),                  'saap_b_left'),
                    ('b' + str(idx + 1),              'saap_b_right'),
                    ('y' + str(len(saap) - idx + 1),  'saap_y_left'),
                    ('y' + str(len(saap) - idx),      'saap_y_right'),
                ], 'SAAP'),
                (bp, [
                    ('b' + str(idx),                  'bp_b_left'),
                    ('b' + str(idx + 1),              'bp_b_right'),
                    ('y' + str(len(bp) - idx + 1),    'bp_y_left'),
                    ('y' + str(len(bp) - idx),        'bp_y_right'),
                ], 'BP'),
            ]:
                # FragPipe: O(1) best fragment-TSV hit for this peptide (most matched ions)
                best = frag_by_pep.get(peptide)
                if best is None:
                    continue
                # build {maxquant ion name: theoretical b/y mass} from the matched fragments
                frags = {}
                for tok in str(best['fragments']).split(';'):
                    tok = tok.strip()
                    if not tok:
                        continue
                    mq = resolve_ion(tok)
                    if mq.startswith(('b', 'y')):
                        frags[mq] = frag_ion_mass(peptide, best['modification_info'], tok.split('^')[0])
                exp_frag[exp][global_i][frag_key] = frags
                for frag_name, col_prefix in mapping:
                    s_df.loc[i, col_prefix + '_frag'] = frag_name
                    if frag_name in frags:
                        s_df.loc[i, col_prefix + '_frag_mass'] = float(frags[frag_name])

        exp_rows[exp].append(s_df)
        exp_count[exp] += len(s_df)

for exp, dfs in exp_rows.items():
    if not dfs:
        continue
    pos_df = pd.concat(dfs)
    pos_df.columns = POS_COLS

    edir     = exp_dir(exp)
    csv_path = os.path.join(edir, 'Position_probability_fragment_ion_data.csv')
    pkl_path = os.path.join(edir, 'Fragment_ion_dict.p')

    pos_df.to_csv(csv_path)
    with open(pkl_path, 'wb') as f:
        pickle.dump(exp_frag[exp], f)
    print(f"  Saved to {csv_path}")
    print(f"  Saved to {pkl_path}")


=== Generating: Position_probability_fragment_ion_data.csv + Fragment_ion_dict.p ===
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/Position_probability_fragment_ion_data.csv
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/Fragment_ion_dict.p
  Saved to /Users/alexmaropakis/Pro

## 4. `SAAP_precursor_reporter_quant.xlsx`

Per-sample SAAP quantification table. Joins precursor + reporter intensities
from `MTP_quant_dict.p`, the minimum PEP for each peptide from MaxQuant
`evidence.txt`, and positional probability from the file produced in the
previous step.


In [45]:
print("=== Generating: SAAP_precursor_reporter_quant.xlsx ===")

ENT_FEATURE = 'weighted_spectral_entropy'   # MSBooster spectral match (higher = better)
RT_FEATURE  = 'delta_RT_loess'              # observed-vs-predicted RT, absolute (lower = better)

def load_pin_features(plex_dir):
    """{(raw_stem, scan): (entropy, delta_rt)} across all *_edited.pin in plex_dir (rank-1).
    Pins live in the same <plex>_1/ dir as psm.tsv (alongside the MSBooster/ subfolder).
    Per scan, keeps the row with the highest entropy (its RT travels with it)."""
    lookup = {}
    for f in glob(plex_dir + '*_edited.pin'):
        header = pd.read_csv(f, sep='\t', nrows=0).columns.tolist()
        if not {ENT_FEATURE, RT_FEATURE, 'SpecId'}.issubset(header):
            continue
        d = pd.read_csv(f, sep='\t', usecols=['SpecId', ENT_FEATURE, RT_FEATURE], low_memory=False)
        for spec, ent, rt in zip(d['SpecId'], d[ENT_FEATURE], d[RT_FEATURE]):
            sid = str(spec)
            if not sid.endswith('_1'):                 # rank-1 PSMs only
                continue
            parts = sid.rsplit('_', 1)[0].rsplit('.', 3)  # drop _rank, then stem.scan.scan.charge
            if len(parts) < 4:
                continue
            try:
                key = (parts[0], int(parts[1]))
            except ValueError:
                continue
            if key not in lookup or ent > lookup[key][0]:
                lookup[key] = (ent, rt)
    return lookup

def spectrum_to_key(spectrum):
    """FragPipe psm.tsv 'Spectrum' (Stem.Scan.Scan.Charge) -> (raw_stem, scan)."""
    s = str(spectrum)
    try:
        return (s.rsplit('.', 3)[0], int(s.split('.')[-3]))
    except (ValueError, IndexError):
        return (None, None)

# Load just-saved positional probabilities across all experiments
pos_prob_all = pd.concat(
    [pd.read_csv(os.path.join(DEPDIR, exp, 'Position_probability_fragment_ion_data.csv'),
                 index_col=0)
     for exp in {get_experiment_label(d) for d in all_datasets}
     if os.path.exists(os.path.join(DEPDIR, exp,
                                    'Position_probability_fragment_ion_data.csv'))],
    ignore_index=True,
)

rows = []
for ds, proj, frag_dir in zip(all_datasets, proj_dir_list, frag_dir_list):
    MTP_quant_dict = compat_load(proj + 'MTP_quant_dict.p')

    # FragPipe: psm.tsv 'Probability' (higher=better) -> PEP-like score (1 - Probability),
    # take min (best) per peptide. Mirrors MaxQuant evidence.txt PEP min lookup.
    psm = pd.read_csv(frag_dir + 'psm.tsv', sep='\t', low_memory=False)
    psm['_pep'] = psm['Peptide'].astype(str).str.strip().str.upper()
    psm['_pepscore'] = 1.0 - psm['Probability']
    pep_lookup = psm.groupby('_pep')['_pepscore'].min().to_dict()

    # MSBooster spectral entropy + RT agreement (localization-rescue features).
    # Resolve best entropy and its RT per (tmt_set, MTP_seq) via Ion_validated_MTP_dict
    # -> psm.tsv Spectrum. Entropy is maximized over a SAAP's PSMs; the chosen PSM's
    # RT travels with it (so the RT belongs to the best-matching spectrum, not a separate one).
    feat_lookup = load_pin_features(frag_dir)
    mtp_dict    = compat_load(proj + 'Ion_validated_MTP_dict.p')
    ev_dict     = compat_load(proj + 'Validation_search_evidence_dict.p')
    feat_by_seq = {}   # (tmt_set, MTP_seq) -> (best_entropy, rt_of_best)
    for s in mtp_dict:
        if not isinstance(mtp_dict[s], dict) or 'mistranslated sequence' not in mtp_dict[s]:
            continue
        ev = ev_dict.get(s)                    # this set's psm.tsv DataFrame
        if ev is None:
            continue
        for k in mtp_dict[s]['mistranslated sequence']:
            seq    = str(mtp_dict[s]['mistranslated sequence'][k]).strip().upper()
            ev_idx = mtp_dict[s]['idx_val_evidence'][k]
            best_ent, best_rt = np.nan, np.nan
            for idx in ev_idx:
                got = feat_lookup.get(spectrum_to_key(ev.iloc[idx]['Spectrum']))
                if got is None:
                    continue
                e, r = got
                if not np.isnan(e) and (np.isnan(best_ent) or e > best_ent):
                    best_ent, best_rt = e, r
            key = (s, seq)
            if key not in feat_by_seq or (not np.isnan(best_ent) and
               (np.isnan(feat_by_seq[key][0]) or best_ent > feat_by_seq[key][0])):
                feat_by_seq[key] = (best_ent, best_rt)

    ds_map = resolve_dataset_key(proj)

    for entry in MTP_quant_dict.values():
        mtp_seq  = str(entry['MTP_seq']).strip().upper()
        bp_seq   = str(entry['BP_seq']).strip().upper()
        aa_sub   = entry['aa_sub']
        tmt_sets = entry['tmt_sets']
        saap_pep = pep_lookup.get(mtp_seq, np.nan)
        bp_pep   = pep_lookup.get(bp_seq,  np.nan)

        for sample, sd in entry['Patient_dict'].items():
            for tmt_set in tmt_sets:
                dataset = ds_map.get(tmt_set, 'Unknown')
                m = pos_prob_all[
                    (pos_prob_all['SAAP']    == mtp_seq) &
                    (pos_prob_all['BP']      == bp_seq)  &
                    (pos_prob_all['TMT set'] == tmt_set) &
                    (pos_prob_all['Dataset'] == dataset)
                ]
                pos_prob = m.iloc[0]['Positional probability'] if not m.empty else np.nan
                spec_ent, delta_rt = feat_by_seq.get((tmt_set, mtp_seq), (np.nan, np.nan))

                rows.append({
                    'Dataset':                dataset,
                    'Experiment':             get_experiment_label(dataset),
                    'MTP_seq':                mtp_seq,
                    'BP_seq':                 bp_seq,
                    'aa_sub':                 aa_sub,
                    'tmt_set':                tmt_set,
                    'Positional_probability': pos_prob,
                    'spectral_entropy':       spec_ent,
                    'delta_RT':               delta_rt,
                    'SAAP_PEP':               saap_pep,
                    'BP_PEP':                 bp_pep,
                    'MTP_PrecInt':            entry['MTP_PrecInt'].get(tmt_set),
                    'BP_PrecInt':             entry['BP_PrecInt'].get(tmt_set),
                    'Prec_RAAS':              entry['Prec_ratio'].get(tmt_set),
                    'Sample':                 sample,
                    'Sample Type':            classify_sample(sample),
                    'MTP_ReportInt':          sd['MTP_ReportInt'],
                    'BP_ReportInt':           sd['BP_ReportInt'],
                    'Reporter_RAAS':          sd['Reporter_ratio'],
                    'BP_ReportInt_Norm':      sd['BP_ReportInt_Norm'],
                    'MTP_ReportInt_Norm':     sd['MTP_ReportInt_Norm'],
                })

saap_df = pd.DataFrame(rows)
saap_df.insert(0, 'Row Number', range(len(saap_df)))
print(f"  spectral_entropy coverage: {saap_df['spectral_entropy'].notna().mean():.1%} of rows")
print(f"  delta_RT coverage:         {saap_df['delta_RT'].notna().mean():.1%} of rows")

for exp, grp in saap_df.groupby('Experiment'):
    grp = grp.drop(columns='Experiment').reset_index(drop=True)
    grp['Row Number'] = range(len(grp))
    out = os.path.join(exp_dir(exp), 'SAAP_precursor_reporter_quant.xlsx')
    grp.to_excel(out, index=False)
    print(f"  Saved to {out}  ({len(grp):,} rows)")

=== Generating: SAAP_precursor_reporter_quant.xlsx ===
  spectral_entropy coverage: 100.0% of rows
  delta_RT coverage:         100.0% of rows
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/SAAP_precursor_reporter_quant.xlsx  (18,910 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/SAAP_precursor_reporter_quant.xlsx  (8,960 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/SAAP_precursor_reporter_quant.xlsx  (4,020 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/SAAP_precursor_reporter_quant.xlsx  (25,424 rows)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Tsumagari_2023/SAAP_precursor_reporter_quant.xlsx  (8,349 rows)


## 5. `MTP_sequences.fasta`

In [46]:
print("=== Generating: MTP_sequences.fasta ===")

exp_lines = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    # Load the MTP quantification dictionary for the dataset and extract MTP sequences and amino acid substitutions,
    # organizing them by experiment for FASTA output
    MTP_dict = compat_load(proj + 'MTP_quant_dict.p')
    exp = get_experiment_label(ds)
    exp_lines.setdefault(exp, [])
    for idx, entry in MTP_dict.items():
        mtp_seq = str(entry['MTP_seq']).strip().upper()
        aa_sub  = str(entry['aa_sub']).replace(',', '_').replace(' ', '')
        exp_lines[exp].extend([f">{ds}_MTP_{idx}_AAsub_{aa_sub}", mtp_seq])

for exp, lines in exp_lines.items():
    # For each experiment, save the compiled MTP sequences and their headers to a FASTA file
    out = os.path.join(exp_dir(exp), 'MTP_sequences.fasta')
    with open(out, 'w') as f:
        f.write('\n'.join(lines))
    print(f"  Saved to {out}  ({len(lines)//2:,} sequences)")

=== Generating: MTP_sequences.fasta ===
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/MTP_sequences.fasta  (896 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/MTP_sequences.fasta  (402 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/MTP_sequences.fasta  (1,589 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/MTP_sequences.fasta  (1,891 sequences)
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Tsumagari_2023/MTP_sequences.fasta  (759 sequences)


## 6. `Modified_peptide_filter_dict_DP2valSAAP.p`

Funnel of peptide counts from raw evidence → DP → PTM → AAS → high-confidence
→ ion-validated. Used to render the filter waterfall plot.


In [47]:
print("=== Generating: Modified_peptide_filter_dict_DP2valSAAP.p ===")

FILTER_FILES = [
    # List of files to load for each dataset, along with the key to extract 
    # from the loaded dictionary and a description of the type of samples they contain
    ('DP_search_evidence_dict.p', 'main', 'Raw file'),
    ('DP_dict.p',                 'dp',   'Raw file'),
    ('PTM_dict.p',                'ptm',  'Raw file'),
    ('MTP_dict.p',                'mtp',  'Raw file'),
    ('qMTP_dict.p',               'qmtp', 'mistranslated sequence'),
    ('Ion_validated_MTP_dict.p',  'ion',  'Raw file'),
]

exp_data = {}
for ds, proj, samples in zip(all_datasets, proj_dir_list, samples_list):
    # Load the specified filter files for the dataset, 
    # extracting the relevant dictionaries based on the provided keys, 
    # and compile counts of peptides passing each filter for each sample
    parts = {key: compat_load(proj + fname) for fname, key, _ in FILTER_FILES}

    rows = [[s] + [len(parts[key][s][col]) for _, key, col in FILTER_FILES]
            for s in samples]
    df = pd.DataFrame(rows, columns=[
        'TMT set', 'Main peptides', 'DP', 'PTM', 'AAS',
        'High-confidence', 'Validated',
    ])
    df['Dataset'] = ds
    exp_data.setdefault(get_experiment_label(ds), {})[ds] = df

for exp, ds_dfs in exp_data.items():
    # For each experiment, save the compiled filter data for all datasets to a pickle file
    out = os.path.join(exp_dir(exp), 'Modified_peptide_filter_dict_DP2valSAAP.p')
    with open(out, 'wb') as f:
        pickle.dump(ds_dfs, f)
    print(f"  Saved → {out}")


=== Generating: Modified_peptide_filter_dict_DP2valSAAP.p ===
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/Modified_peptide_filter_dict_DP2valSAAP.p
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Tsumagari_2023/Modified_peptide_filter_dict_DP2valSAAP.p


## 7 + 8. `PTM_heatmap_dict.p` and `PTM_heatmap_data.xlsx`

Per-dataset PTM count matrix (rows = PTMs, cols = TMT plex), then the shared
top-N PTMs across all datasets in an experiment, normalized by peptides per
1000 (`Peptides (evidence)` from `dataset_metrics.xlsx`).


In [48]:
print("=== Generating: PTM_heatmap_dict.p + PTM_heatmap_data.xlsx ===")

TOP_N     = 40
N_SAMPLES = 23

def build_ptm_count_df(ptm_dict):
    # Build a DataFrame counting the occurrences of each PTM across samples for a given PTM dictionary,
    # ensuring that all PTMs are included as rows and all samples as columns, with missing values filled with zeros
    master = []
    for v in ptm_dict.values():
        for ptms in v['PTM'].values():
            for p in ptms:
                if p not in master:
                    master.append(p)
    df = pd.DataFrame(index=master, columns=range(1, N_SAMPLES + 1))
    for s, v in ptm_dict.items():
        try:    s_int = int(s[1:])
        except: s_int = s
        cnt = Counter(p for sub in v['PTM'].values() for p in sub)
        for p, c in cnt.items():
            df.loc[p, s_int] = c
    return df.fillna(0).astype(float)

exp_ptm = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    # Load the PTM dictionary for the dataset, build a DataFrame counting PTM occurrences across samples,
    # and store it in a dictionary organized by experiment and dataset for later saving and plotting
    exp = get_experiment_label(ds)
    df  = build_ptm_count_df(compat_load(proj + 'PTM_dict.p'))
    df['avg'] = df.mean(axis=1)
    df.sort_values('avg', ascending=False, inplace=True)
    df.index = [x[0].upper() + x[1:] for x in df.index]
    exp_ptm.setdefault(exp, {})[ds] = df

for exp, ptm_heatmap_dict in exp_ptm.items():
    # For each experiment, save the compiled PTM heatmap data for all datasets to a pickle file,
    # and also compile a DataFrame of the top PTMs across datasets, normalized by the 
    # number of peptides in each dataset, and save it to an Excel file for plotting
    edir = exp_dir(exp)

    pkl_path = os.path.join(edir, 'PTM_heatmap_dict.p')
    with open(pkl_path, 'wb') as f:
        pickle.dump(ptm_heatmap_dict, f)
    print(f"  Saved → {pkl_path}")

    metrics = pd.read_excel(os.path.join(edir, 'dataset_metrics.xlsx'))

    top = None
    for df in ptm_heatmap_dict.values():
        # Identify the top N PTMs in each dataset and find the intersection of 
        # these top PTMs across all datasets in the experiment,
        # to focus the heatmap on the most commonly observed PTMs
        s   = set(df.index[:TOP_N])
        top = s if top is None else top & s
    if not top:
        print(f"  WARNING: no shared top-{TOP_N} PTMs in '{exp}' — skipping data file")
        continue

    plot_df = pd.DataFrame({
        # Compile a DataFrame containing the average counts of the top PTMs for each dataset,
        # normalized by the number of peptides in each dataset to account for differences in dataset size
        ds: ptm_heatmap_dict[ds].loc[list(top), 'avg']
        for ds in ptm_heatmap_dict
    }).astype(float)

    for ds in plot_df.columns:
        # Normalize the average PTM counts for each dataset by the number of peptides (in thousands) in that dataset,
        # to allow for fair comparison of PTM prevalence across datasets of different sizes
        n = metrics.loc[metrics['Dataset'] == ds, 'Peptides (evidence)'].values
        if len(n):
            plot_df[ds] = plot_df[ds] / (n[0] / 1000)

    xlsx_path = os.path.join(edir, 'PTM_heatmap_data.xlsx')
    plot_df.to_excel(xlsx_path)
    print(f"  Saved to {xlsx_path}")


=== Generating: PTM_heatmap_dict.p + PTM_heatmap_data.xlsx ===


/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_94214/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.fillna(0).astype(float)
/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_94214/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.fillna(0).astype(float)
/var/folders/g1/2zxfph5533g5zjz997szslmw0000gn/T/ipykernel_94214/1587355368.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/PTM_heatmap_data.xlsx
  Saved → /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Tsumagari_2023/PTM_heatmap_dict.p
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Tsumagari_2023/PTM_heatmap_data.xlsx


## 9. `fragments_per_saap_4barplot_allDS.xlsx`

For each validated SAAP, counts fragment ions (b/y) crossing the substituted
residue across all validation-search evidence scans. Bins per experiment into
{0, 1, 2-5, 6-10, >10}; bins 0 and 1 are flagged "Used = No".

Note: this step also writes `fragment_evidence` back into each
`Validated_MTP_dict.p` for downstream use.


In [49]:
print("=== Generating: fragments_per_saap_4barplot_allDS.xlsx ===")

CATS = ['0', '1', '2-5', '6-10', '>10']

def n_frags_over_mtp(matches, mtp, sub_idx):
    # FragPipe fragments: 'nterm{n}'=b{n}, 'cterm{n}'=y{n}, '^z' charge.
    # Count b/y ions spanning the substitution site (same span logic as MaxQuant version).
    count = 0
    L = len(mtp)
    for f in matches:
        ion = f.split('^')[0]
        if 'nterm' in ion:
            if int(ion[5:]) > sub_idx:
                count += 1
        elif 'cterm' in ion:
            if L - int(ion[5:]) <= sub_idx:
                count += 1
    return count

exp_counts = {}
for ds, proj, samples, frag_dir in zip(all_datasets, proj_dir_list, samples_list, frag_dir_list):
    exp = get_experiment_label(ds)
    exp_counts.setdefault(exp, [0, 0, 0, 0, 0])

    val_path  = proj + 'Validated_MTP_dict.p'
    mtp_dict  = compat_load(val_path)

    # FragPipe: same peptide-keyed best-hit index used in step 2/3 (load_frag_index) --
    # reused as-is here instead of re-deriving a per-scan lookup
    frag_by_pep = load_frag_index(frag_dir)

    print(f"  Processing {ds}")

    for s in samples:
        if s not in mtp_dict:
            continue
        mtp_dict[s]['fragment_evidence'] = {}

        for k in mtp_dict[s]['aa subs']:
            seq     = mtp_dict[s]['mistranslated sequence'][k]
            bp      = mtp_dict[s]['DP Base Sequence'][k]
            sub_idx = next(i for i, x in enumerate(bp) if seq[i] != x)
            best    = 0

            frag_entry = frag_by_pep.get(seq)
            if frag_entry is not None:
                frag_str = frag_entry['fragments']
                if isinstance(frag_str, str) and frag_str.strip():
                    best = n_frags_over_mtp([t for t in frag_str.split(';') if t], seq, sub_idx)
            mtp_dict[s]['fragment_evidence'][k] = best

    with open(val_path, 'wb') as f:
        pickle.dump(mtp_dict, f)

    for s in samples:
        if s not in mtp_dict:
            continue
        for n in mtp_dict[s].get('fragment_evidence', {}).values():
            if   n == 0:  exp_counts[exp][0] += 1
            elif n == 1:  exp_counts[exp][1] += 1
            elif n <= 5:  exp_counts[exp][2] += 1
            elif n <= 10: exp_counts[exp][3] += 1
            else:         exp_counts[exp][4] += 1

for exp, counts in exp_counts.items():
    out = os.path.join(exp_dir(exp), 'fragments_per_saap_4barplot_allDS.xlsx')
    df  = pd.DataFrame(zip(CATS, counts), columns=['Bin', 'Count'])
    df['Used'] = ['No' if b in ('0', '1') else 'Yes' for b in df['Bin']]
    df.to_excel(out)
    print(f"  Saved to {out}")


=== Generating: fragments_per_saap_4barplot_allDS.xlsx ===
  Processing acgb1
  Processing acgb2
  Processing acgb3
  Processing acgb4
  Processing acgb5
  Processing fcb1
  Processing fcb2
  Processing fcb3
  Processing fcb4
  Processing fcb5
  Processing Aorta
  Processing Brain
  Processing Heart
  Processing Kidney
  Processing Liver
  Processing Lung
  Processing Muscle
  Processing Skin
  Processing pooled
  Processing cortex_1
  Processing cortex_2
  Processing hippocampus_1
  Processing hippocampus_2
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/acg/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Ping_2018/fc/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Takasugi_2024/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /Users/alexmaropakis/Projects/BrainDecode/Dependencies/Bai_2020/fragments_per_saap_4barplot_allDS.xlsx
  Saved to /User

In [50]:
print("=== Generating genome_substr_dict.p files ===")

GENOME_SUBSTR_KEYS = [
    '1-frame genome substring',
    '2-frame genome substring',
    '3-frame genome substring',
    '4-frame genome substring',
    '5-frame genome substring',
    '6-frame genome substring',
]

for proj_dir in proj_dir_list:
    dp_path = os.path.join(proj_dir, 'DP_dict.p')
    if not os.path.exists(dp_path):
        print(f"Skipping missing DP_dict.p: {dp_path}")
        continue

    print(f"Processing: {dp_path}")
    try:
        dp_dict = pickle.load(open(dp_path, 'rb'))
        genome_substr_dict = {}
        all_mtps = set()
        frame_hits = {
            1: set(),
            2: set(),
            3: set(),
            4: set(),
            5: set(),
            6: set(),
        }

        # dp_dict structure:
        # dp_dict[sample][field][index]
        for sample in dp_dict.keys():
            mt_seq_dict = dp_dict[sample]['mistranslated sequence']
            for idx, seq_list in mt_seq_dict.items():
                all_mtps.update(seq_list)
                for frame in range(1, 7):
                    key = f'{frame}-frame genome substring'
                    if key not in dp_dict[sample]:
                        continue
                    hit_list = dp_dict[sample][key].get(idx, [])
                    for seq_i, seq in enumerate(seq_list):
                        if (
                            seq_i < len(hit_list)
                            and hit_list[seq_i] == True
                        ):
                            frame_hits[frame].add(seq)

        genome_substr_dict = {
            'MTPs': list(all_mtps),

            'all_frame1_seqs': list(frame_hits[1]),
            'N_all_frame1': len(frame_hits[1]),

            'all_frame2_seqs': list(frame_hits[2]),
            'N_all_frame2': len(frame_hits[2]),

            'all_frame3_seqs': list(frame_hits[3]),
            'N_all_frame3': len(frame_hits[3]),

            'all_frame4_seqs': list(frame_hits[4]),
            'N_all_frame4': len(frame_hits[4]),

            'all_frame5_seqs': list(frame_hits[5]),
            'N_all_frame5': len(frame_hits[5]),

            'all_frame6_seqs': list(frame_hits[6]),
            'N_all_frame6': len(frame_hits[6]),

            # backwards compatibility with older decode notebooks
            'all_1frame_seqs': list(frame_hits[1]),
            'N_all_1frame': len(frame_hits[1]),

            'all_3frame_seqs': list(frame_hits[3]),
            'N_all_3frame': len(frame_hits[3]),

            'all_6frame_seqs': list(frame_hits[6]),
            'N_all_6frame': len(frame_hits[6]),
        }

        out_path = os.path.join(proj_dir, 'genome_substr_dict.p')
        pickle.dump(
            genome_substr_dict,
            open(out_path, 'wb')
        )
        print(f"Saved: {out_path}")

    except Exception as e:
        print(f"ERROR processing {proj_dir}")
        print(str(e))

print("Finished generating genome_substr_dict.p files.")

=== Generating genome_substr_dict.p files ===
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b1/DP_dict.p


Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b1/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b2/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b2/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b3/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b3/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b4/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b4/genome_substr_dict.p
Processing: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b5/DP_dict.p
Saved: /Users/alexmaropakis/Projects/Project_BrainDecode/Analysis_Inputs/Ping_2018/acg/b5/genome_substr_dict.p
Processing: /Users/alexma

In [51]:
print("All dependencies generated.")

All dependencies generated.
